In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import joblib

In [ ]:
# Load dataset
data = pd.read_csv('pred.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8547 entries, 0 to 8546
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   id                    8547 non-null   int64 
 1   title                 8547 non-null   object
 2   revenue               8547 non-null   int64 
 3   runtime               8547 non-null   int64 
 4   budget                8547 non-null   int64 
 5   genres                8547 non-null   object
 6   production_companies  8547 non-null   object
 7   production_countries  8547 non-null   object
 8   directors             8547 non-null   object
 9   writers               8547 non-null   object
dtypes: int64(4), object(6)
memory usage: 667.9+ KB


In [ ]:
data.head()

,id,title,revenue,runtime,budget,genres,production_companies,production_countries,directors,writers
0,27205,Inception,825532764,148,160000000,208,3657,"United Kingdom, United States of America",1038,1402
1,157336,Interstellar,701729206,169,165000000,340,3656,"United Kingdom, United States of America",1038,3911
2,155,The Dark Knight,1004558444,152,185000000,939,1692,"United Kingdom, United States of America",1038,3913
3,19995,Avatar,2923706026,162,237000000,48,2024,"United States of America, United Kingdom",2360,3143
4,24428,The Avengers,1518815515,143,220000000,1621,4076,United States of America,3001,4019


In [ ]:
# Initialize LabelEncoders for each categorical column
le_genres = LabelEncoder()
le_production_companies = LabelEncoder()
le_directors = LabelEncoder()
le_writers = LabelEncoder()

# Apply encoding to each categorical column
data['genres'] = le_genres.fit_transform(data['genres'].astype(str))
data['production_companies'] = le_production_companies.fit_transform(data['production_companies'].astype(str))
data['directors'] = le_directors.fit_transform(data['directors'].astype(str))
data['writers'] = le_writers.fit_transform(data['writers'].astype(str))

In [ ]:
X = data[['runtime', 'budget', 'genres', 'production_companies', 'directors', 'writers']]  # Features
y = data['revenue']  # Target variable

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()

# Fit on training data and transform
X_train_scaled = scaler.fit_transform(X_train)

# Transform the test data (without fitting again)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Initialize and train the model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

RandomForestRegressor(random_state=42)

In [ ]:
# Save the model to a file
joblib.dump(model, 'movie_revenue_predictor.pkl')
joblib.dump(scaler, 'scaler.pkl')  # If you used scaling

['scaler.pkl']

In [ ]:
#joblib.load('movie_revenue_predictor.pkl')
#joblib.load('scaler.pkl')

import joblib

uploaded_model_path = 'movie_revenue_predictor.pkl'

# Attempting to load the model to inspect its properties
loaded_model = joblib.load(uploaded_model_path)

# Checking the attributes of the loaded model
#dir(loaded_model)

In [ ]:
# Save the LabelEncoders after training
joblib.dump(le_genres, 'le_genres.pkl')
joblib.dump(le_production_companies, 'le_production_companies.pkl')
joblib.dump(le_directors, 'le_directors.pkl')
joblib.dump(le_writers, 'le_writers.pkl')

['le_writers.pkl']

In [ ]:
from sklearn.preprocessing import LabelEncoder
import joblib
import numpy as np

def encode_with_fallback(encoder, value, fallback_value=-1):
    """Encode with fallback for unseen labels."""
    try:
        return encoder.transform([value])[0]  # Try encoding
    except ValueError:  # If unseen label is encountered
        return fallback_value  # Return a fallback value for unseen labels

def predict_revenue(input_data):
    # Load the model and scaler
    model = joblib.load('movie_revenue_predictor.pkl')
    scaler = joblib.load('scaler.pkl')

    # Load the LabelEncoders used during training
    le_genres = joblib.load('le_genres.pkl')
    le_production_companies = joblib.load('le_production_companies.pkl')
    le_directors = joblib.load('le_directors.pkl')
    le_writers = joblib.load('le_writers.pkl')

    # Encode categorical features with fallback
    input_data_encoded = [
        input_data[0],  # runtime (numeric)
        input_data[1],  # budget (numeric)
        encode_with_fallback(le_genres, input_data[2]),  # genres (encoded)
        encode_with_fallback(le_production_companies, input_data[3]),  # production_companies (encoded)
        encode_with_fallback(le_directors, input_data[4]),  # directors (encoded)
        encode_with_fallback(le_writers, input_data[5]),  # writers (encoded)
    ]

    # Scale the input data using the same scaler used during training
    input_data_scaled = scaler.transform([input_data_encoded])  # Transform input_data (it should be a list of feature values)

    # Predict revenue
    predicted_revenue = model.predict(input_data_scaled)

    return predicted_revenue[0]
